## Training the Final Chess Data with DecisionTree, RandomForest and XGBoost

In [19]:
import os
import pandas as pd
import numpy as np
import joblib
import sklearn
from sklearn.metrics import confusion_matrix, classification_report

#### Loading the Data

In [8]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir,"data/final_chess.csv"))

In [10]:
games

,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO,EloDiff,AvgElo,...,h2h_WinDiff,h2h_WinRateDiff,DaysSinceLastGameDiff,ECO_Games,ECO_WhiteWin,ECO_BlackWin,ECO_Draw,ECO_WhiteWinRate,ECO_BlackWinRate,ECO_DrawRate
0,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13,-69.0,2171.5,...,0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000
1,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46,-272.0,2273.0,...,0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000
2,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42,-20.0,2246.0,...,0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000
3,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46,103.0,2263.5,...,0,0.0,0,1,0,1,0,0.000000,1.000000,0.000000
4,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17,40.0,2194.0,...,0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3255651,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11,-170.0,2397.0,...,0,0.0,0,8963,3442,3250,2271,0.384023,0.362602,0.253375
3255652,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28,90.0,2433.0,...,0,0.0,0,9793,3964,3997,1832,0.404779,0.408149,0.187072
3255653,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06,146.0,2393.0,...,0,0.0,0,20988,9639,6895,4454,0.459262,0.328521,0.212217
3255654,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40,-96.0,2503.0,...,0,0.0,0,45159,19568,17785,7806,0.433313,0.393831,0.172856


#### Converting the Data for Training

In [ ]:
X = games.drop(columns=["White", "Black", "Result", "Date", "Site", "ECO"])
y = games["Result"].map({1: 2, 0.5: 1, 0:0}) #sklearn only considers integers 

"""
0 = Black win
1 = Draw
2 = White win
"""

In [43]:
n = len(games)
s1 = int(0.7 * n)
s2 = int(0.85 * n)

In [44]:
X_train = X.iloc[:s1]
y_train = y.iloc[:s1]
X_val = X.iloc[s1:s2]
y_val = y.iloc[s1:s2]
X_test = X.iloc[s2:]
y_test = y.iloc[s2:]

#### Decision Tree

In [45]:
from sklearn.tree import DecisionTreeClassifier

In [46]:
dtc = DecisionTreeClassifier()

In [47]:
dtc.get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [48]:
dtc.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [58]:
y_pred = dtc.predict(X_val)
y_pred = dtc.predict(X_test)

In [59]:
confusion_matrix(y_test, y_pred)

array([[ 85081,  33681,  61682],
       [ 31858,  27758,  35428],
       [ 64349,  39647, 108865]])

In [61]:
classification_report(y_test, y_pred).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.47      0.47      0.47    180444',
 '           1       0.27      0.29      0.28     95044',
 '           2       0.53      0.51      0.52    212861',
 '',
 '    accuracy                           0.45    488349',
 '   macro avg       0.42      0.43      0.42    488349',
 'weighted avg       0.46      0.45      0.46    488349']

In [62]:
X.columns

Index(['WhiteElo', 'BlackElo', 'EloDiff', 'AvgElo', 'HigherRatedPlayer',
       'WhiteGames', 'WhiteWin', 'WhiteLoss', 'WhiteDraw', 'BlackGames',
       'BlackWin', 'BlackLoss', 'BlackDraw', 'WhiteWinRate', 'WhiteLossRate',
       'WhiteDrawRate', 'BlackWinRate', 'BlackLossRate', 'BlackDrawRate',
       'WLast10', 'BLast10', 'WasWGames', 'WasWWin', 'WasWLoss', 'WasWDraw',
       'WasBGames', 'WasBWin', 'WasBLoss', 'WasBDraw', 'BasWGames', 'BasWWin',
       'BasWLoss', 'BasWDraw', 'BasBGames', 'BasBWin', 'BasBLoss', 'BasBDraw',
       'WasWWinRate', 'WasWLossRate', 'WasWDrawRate', 'WasBWinRate',
       'WasBLossRate', 'WasBDrawRate', 'BasWWinRate', 'BasWLossRate',
       'BasWDrawRate', 'BasBWinRate', 'BasBLossRate', 'BasBDrawRate',
       'h2h_games', 'h2h_Player1Win', 'h2h_Player2Win', 'h2h_Draw',
       'h2h_Player1WinRate', 'h2h_Player2WinRate', 'h2h_DrawRate', 'h2h_last5',
       'WhiteDaysSinceLastGame', 'BlackDaysSinceLastGame', 'Last10Diff',
       'GamesDiff', 'WinDiff', 'WinRa

In [76]:
pd.DataFrame(dtc.feature_importances_, index=X.columns).sort_values(by=0).head(50)

,0
HigherRatedPlayer,0.000107
h2h_Draw,0.001110
h2h_Player1Win,0.001136
h2h_Player2Win,0.001165
h2h_Player2WinRate,0.001597
h2h_Player1WinRate,0.001651
h2h_WinDiff,0.001987
h2h_WinRateDiff,0.002217
h2h_games,0.002615
h2h_DrawRate,0.002708
